# 00 — ChEMBL Setup & EDA

Phase 0 notebook for **CRC_Inhibitor_ML**. Goals:

1. Connect to the local ChEMBL SQLite database
2. Confirm our four CRC oncogene targets exist with the expected ChEMBL IDs
3. Pull every IC50 bioactivity record joined with SMILES for those targets
4. Basic EDA on the raw pull (counts per target, units sanity, pIC50 distribution, missingness)
5. Save the raw extract to `data/raw/`

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

CHEMBL_DB_PATH = r"E:\ml_data\chembl\chembl_37\chembl_37_sqlite\chembl_37.db"
PROJECT_ROOT   = Path(r"C:\Users\palla\OneDrive\Documents\Coding Projects\CRC_Inhibitor_ML")
RAW_DATA_DIR   = PROJECT_ROOT / "data" / "raw"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Four CRC oncogene targets (verified against ChEMBL 37 target_dictionary)
TARGETS = {
    "CHEMBL2189121": "KRAS",      # GTPase KRas — covers WT + mutants (incl. G12C) pooled
    "CHEMBL5145":    "BRAF",      # Serine/threonine-protein kinase B-raf
    "CHEMBL203":     "EGFR",      # Epidermal growth factor receptor
    "CHEMBL4005":    "PIK3CA",    # PI3K-alpha catalytic subunit
}

print(f"ChEMBL DB:    {CHEMBL_DB_PATH}")
print(f"Raw out dir:  {RAW_DATA_DIR}")
print(f"Targets:      {list(TARGETS.values())}")

## Cell 1 — connect and confirm we can read tables

ChEMBL has ~80 tables. We only care about a handful:

| Table | Why we care |
|---|---|
| `target_dictionary` | Maps `target_chembl_id` (e.g. CHEMBL203) → biological target metadata |
| `assays` | Assay metadata — assay type, confidence score, target FK |
| `activities` | The actual measurements (IC50 values etc.), one row per measurement |
| `molecule_dictionary` | One row per molecule (`molregno` is the internal PK) |
| `compound_structures` | SMILES + InChI for each molecule, joined on `molregno` |

In [ ]:
con = sqlite3.connect(CHEMBL_DB_PATH)
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;",
    con,
)
print(f"{len(tables)} tables in DB")
tables.head(30)

## Cell 2 — confirm our four targets exist

Each target's ChEMBL ID should resolve to a known protein with `target_type='SINGLE PROTEIN'` in *Homo sapiens*. If a row is missing, the ID has been renamed in a newer ChEMBL release and we'd need to update `TARGETS`.

In [ ]:
target_ids = list(TARGETS.keys())
target_list_sql = ", ".join(f"'{tid}'" for tid in target_ids)

targets_df = pd.read_sql(
    f"""
    SELECT chembl_id AS target_chembl_id, pref_name, target_type, organism
    FROM target_dictionary
    WHERE chembl_id IN ({target_list_sql})
    """,
    con,
)
targets_df

## Cell 3 — pull the bioactivity data

The big query. Pulls one row per IC50 measurement against any of our four targets, joined with the molecule's canonical SMILES. We keep some extra columns (assay_type, confidence_score, pchembl_value) for Phase 1 filtering; not all of them will survive curation.

- `standard_type = 'IC50'` — only IC50 measurements (we'll filter further in Phase 1)
- `pchembl_value` — ChEMBL's pre-computed `-log10(activity in M)` — this is essentially our pIC50 label, when it's available
- We don't filter by `confidence_score` or `assay_type` here — we want to see the full pull first and decide cutoffs based on the EDA

In [ ]:
bioact_df = pd.read_sql(
    f"""
    SELECT
        md.chembl_id            AS molecule_chembl_id,
        cs.canonical_smiles,
        act.standard_type,
        act.standard_relation,
        act.standard_value,
        act.standard_units,
        act.pchembl_value,
        a.assay_type,
        a.confidence_score,
        td.chembl_id            AS target_chembl_id,
        td.pref_name            AS target_name
    FROM activities act
    JOIN assays              a  ON act.assay_id    = a.assay_id
    JOIN target_dictionary   td ON a.tid           = td.tid
    JOIN molecule_dictionary md ON act.molregno    = md.molregno
    JOIN compound_structures cs ON md.molregno     = cs.molregno
    WHERE td.chembl_id IN ({target_list_sql})
      AND act.standard_type   = 'IC50'
      AND act.standard_value  IS NOT NULL
      AND cs.canonical_smiles IS NOT NULL
    """,
    con,
)

print(f"Pulled {len(bioact_df):,} rows")
bioact_df.head()

## Cell 4 — EDA: counts per target

Sanity check on target balance. KRAS G12C is a young target — far fewer measurements than EGFR (which has been studied since the 90s). Expect orders-of-magnitude imbalance.

In [ ]:
counts = bioact_df["target_name"].value_counts()
print(counts)

## Cell 5 — EDA: standard units

`standard_units` should mostly be `nM`. Anything else (uM, M, %, etc.) is either a different scale that needs unit conversion or a non-IC50 measurement that slipped through the filter — Phase 1 cleanup will handle it.

In [ ]:
bioact_df["standard_units"].value_counts(dropna=False).head(10)

## Cell 6 — EDA: pIC50 distribution

`pchembl_value` is the pre-computed pIC50 (-log10 of activity in molar units). A pIC50 of 9 means IC50 = 1 nM (very potent), 6 = 1 µM (modest), 3 = 1 mM (essentially inactive). Drug-like inhibitors are usually 6-10. A bimodal distribution is common — active vs inactive screening hits.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bioact_df["pchembl_value"].hist(bins=50, ax=ax)
ax.set_xlabel("pIC50 (pchembl_value)")
ax.set_ylabel("count")
ax.set_title("Raw pIC50 distribution across all 4 CRC targets")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True)
for ax, (chembl_id, name) in zip(axes.flat, TARGETS.items()):
    subset = bioact_df.loc[bioact_df["target_chembl_id"] == chembl_id, "pchembl_value"]
    ax.hist(subset.dropna(), bins=40)
    ax.set_title(f"{name}  (n={subset.notna().sum():,})")
    ax.set_xlabel("pIC50")
plt.tight_layout()
plt.show()

## Cell 7 — EDA: missingness audit

We'll need `pchembl_value` (or `standard_value` + `standard_units` if pchembl is null) for every row that becomes a training label. Rows missing both are unusable.

In [ ]:
missing = bioact_df.isna().mean().sort_values(ascending=False) * 100
missing.round(2)

## Cell 8 — save raw extract

Output goes to `data/raw/` (gitignored). This is the *raw* pull — Phase 1 will produce `data/interim/` with standardized SMILES + harmonized pIC50, then `data/processed/` with featurized graphs and train/val/test splits.

In [ ]:
out_path = RAW_DATA_DIR / "chembl_crc_targets_raw.csv"
bioact_df.to_csv(out_path, index=False)
print(f"Saved {len(bioact_df):,} rows to {out_path}")

In [ ]:
con.close()